# Objectives:

In this notebook, we will clean the data and make it ready for EDA. This process will include, but will not be limited to:
- Dealing with duplicates
- Turning the data in InvoiceDate to datetime64
- Dealing with rows with InvoiceNo starting with "A"
- Removing the "C" in the cancelled invoice numbers then adding a new column to mark cancelled orders
- Dealing with rows with unspecified countries
- Dealing with alphabetic stock codes
- Dealing with rows with negative quantity
- Dealing with rows with a negative unit price
- Dealing with rows with a unit price of zero
- Deciding what to do with missing values in the CustomerID and Description columns
- Seeing if missing values in the Description column can be replaced by stock codes

# Code:

## Initialization:

In [1]:
import os
os.chdir('..')

import numpy as np
import pandas as pd

from src.cleaning import remove_bookkeeping_stockcodes
from src.cleaning import remove_bookkeeping_rows
from src.cleaning import remove_zero_price_rows

from src.transformers import cancelled_column_creation
from src.transformers import normalize_invoice_numbers
from src.transformers import standardize_descriptions

from src.calculators import proportion

In [2]:
raw_df = pd.read_csv('data/raw/Online_Retail.csv', encoding='ISO-8859-1')

In [3]:
df = raw_df.copy()

## Structural Data Cleaning:

### Duplicates:

The data contains 5,268 duplicated rows.\
While it is theoretically possible for a customer to repurchase the same item with the same quantity in the same invoice at the exact same timestamp, the probability of this occurring is extremely low for it to happen in 5268 transactions (~0.1 of the dataset).\
It is more likely that these are system-generated errors or data-entry errors rather than genuine repeated transactions. Therefore, the duplicate rows will be removed.

In [4]:
df = df.drop_duplicates()

### Datatypes:

In [5]:
df.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

The InvoiceDate column is stored as a string data type and not datetime64. Let's change that:

In [6]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

C:\Users\user\AppData\Local\Temp\ipykernel_28440\3633860036.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


In [7]:
df['InvoiceDate'].sample(10)

541578   2011-12-09 10:51:00
78393    2011-02-01 12:27:00
65629    2011-01-20 18:08:00
415823   2011-10-24 17:03:00
162062   2011-04-18 15:05:00
429961   2011-10-31 14:09:00
263272   2011-07-14 11:30:00
376411   2011-10-04 14:42:00
27169    2010-12-13 09:35:00
477030   2011-11-17 13:46:00
Name: InvoiceDate, dtype: datetime64[us]

---

## Non-transaction entires:

### Bookkeeping Stock Codes:

The majority of alphabetic stock codes don't represent real transactions, but rather system-level bookkeepings like accounting adjustments, shipping fees, or inventory updates.\
Since these types of adjustments are not part of this project's analysis goals, they should be removed from the data.

In [8]:
df = remove_bookkeeping_stockcodes(df)

In [9]:
len(df)

533652

---

### Bookkeeping Descriptions:

There are some transactions containing descriptions that represent system-level bookkeeping and have a normal stock code.\
From the previous notebook, it was discovered that those transactions are identified when the following conditions are present:
- The quantity is negative
- The transaction was not cancelled
- The customer id is missing
- The description is not missing

So again, since bookkeeping entries do not align with our analysis goals, they should be removed.

In [10]:
df = remove_bookkeeping_rows(df)

In [11]:
len(df)

533183

For confirmation let's check if the data contains some system-level bookkeepings by checking some random known bookkeeping descriptions:

In [12]:
bookkeeping_descriptions = ['post', 'check', 'damages', '?', 'amazonfee', 'found', 'lost', 'b', 'postage', 'amazon', 'manual', 'm']

In [13]:
df[df['Description'].str.lower().isin(bookkeeping_descriptions)].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
6391,536941,22734,amazon,20,2010-12-03 12:08:00,0.0,NaN,United Kingdom
6392,536942,22139,amazon,15,2010-12-03 12:08:00,0.0,NaN,United Kingdom
15651,537639,22734,amazon,30,2010-12-07 15:29:00,0.0,NaN,United Kingdom
24290,538348,22734,amazon,30,2010-12-10 14:59:00,0.0,NaN,United Kingdom
38261,539494,21479,?,752,2010-12-20 10:36:00,0.0,NaN,United Kingdom
39047,539611,85135B,Found,53,2010-12-20 14:33:00,0.0,NaN,United Kingdom
51757,540673,21644,found,144,2011-01-10 16:04:00,0.0,NaN,United Kingdom
51758,540674,22837,Found,26,2011-01-10 16:05:00,0.0,NaN,United Kingdom
51759,540675,20748,Found,40,2011-01-10 16:05:00,0.0,NaN,United Kingdom
115807,546139,84988,?,3000,2011-03-09 16:35:00,0.0,NaN,United Kingdom


Such entries still exist in the dataset, and they seem to be associated with zero-price transactions.

---

### Zero-price Transactions:

We've just discovered that unit prices of zero might be associated with system-level bookkeepings.\
Let's check the unique descriptions of transactions with a unit price of zero:

In [14]:
zero_price_rows = df[df['UnitPrice'] == 0]
zero_price_rows['Description'].value_counts().head(10)

Description
check                             39
found                             25
adjustment                        14
FRENCH BLUE METAL DOOR SIGN 1      9
amazon                             8
FRENCH BLUE METAL DOOR SIGN 8      8
Found                              8
FRENCH BLUE METAL DOOR SIGN 4      7
FRENCH BLUE METAL DOOR SIGN No     7
OWL DOORSTOP                       7
Name: count, dtype: int64

Transactions with zero unit price represent promotional items, replacements for a returned item, or bookkeeping entries. Since they do not generate revenue and would distort price-based analysis, they should be removed from the dataset.

In [15]:
df = remove_zero_price_rows(df)

In [16]:
len(df)

531172

Again, for confirmation let's check if the data contains some system-level bookkeepings by checking some random known bookkeeping descriptions.

In [17]:
df[df['Description'].str.lower().isin(bookkeeping_descriptions)].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


There are none left.

---

## Transaction-level Data Cleaning:

### InvoiceNo Cancellation:

The invoice numbers starting with the letter "C" represent cancellations, but they are still invoice numbers. Let's remove the "C" letter from invoices:

But before we remove the "C", let's create a new column to represent whether an invoice is cancelled or not.

In [18]:
df['Cancelled'] = cancelled_column_creation(df['InvoiceNo'])
df.sample(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Cancelled
404122,571669,84029E,RED WOOLLY HOTTIE WHITE HEART.,3,2011-10-18 13:11:00,4.25,17758.0,United Kingdom,False
454225,575516,22178,VICTORIAN GLASS HANGING T-LIGHT,18,2011-11-10 10:45:00,1.95,17340.0,United Kingdom,False
98071,544667,85099B,JUMBO BAG RED RETROSPOT,100,2011-02-22 15:09:00,1.65,17511.0,United Kingdom,False
376307,569521,23301,GARDENERS KNEELING PAD KEEP CALM,2,2011-10-04 14:33:00,1.65,16474.0,United Kingdom,False
279987,561369,90214E,"LETTER ""E"" BLING KEY RING",1,2011-07-26 16:22:00,0.83,NaN,United Kingdom,False


Now, let's remove the "C":

In [19]:
df['InvoiceNo'] = normalize_invoice_numbers(df['InvoiceNo'])

---

### Negative Quantities:

In [20]:
negative_quantity_rows = df[df['Quantity'] < 0]
print('Rows with a negative quantity:', len(negative_quantity_rows))
print('Proportion of transactions with a negative quantity:', proportion(negative_quantity_rows, df))

Rows with a negative quantity: 8668
Proportion of transactions with a negative quantity: 1.63%


~1.6% of the transactions contain a negative quantity.

From the previous notebook, it was discovered that the transactions with a negative quantity are split into three segments:
1. Cancelled transactions (Should remain in the data since negative quantities are justified in this case)
2. Non-cancelled transactions with a missing customer id and description (The proper action is dependent on how much of the data is represented by them)
3. Non-cancelled system-level bookkeepings with a missing customer id (The system-level bookkeepings were already removed)

Let's check how much do the non-cancelled transactions with a negative quantity represent the data:

In [21]:
negative_quantity_notcancelled = negative_quantity_rows[~negative_quantity_rows['Cancelled']]
print('Rows with a negative quantity and don\'t represent a cancelled transaction:', len(negative_quantity_notcancelled))
print('Proportion of transactions with a negative quantity and don\'t represent a cancellation:', proportion(negative_quantity_notcancelled, df))

Rows with a negative quantity and don't represent a cancelled transaction: 0
Proportion of transactions with a negative quantity and don't represent a cancellation: 0.0%


The transactions with a negative quantity and are not cancelled must have been already removed in the above data cleaning.\
Let's confirm:

In [22]:
negative_quantity_rows['Cancelled'].all()

np.True_

In [23]:
df['Cancelled'][df['Quantity'] < 0].all()

np.True_

Now all transactions with a negative quantity are cancelled, and all cancelled transactions contain a negative quantity. Which makes sense.\
Since the cancelled transactions are already flagged with the "Cancelled" column, the negative sign has no use since it should represent item count, not direction.\
Let's remove the negative sign:

In [24]:
df['Quantity'] = df['Quantity'].abs()

---

## Product labelling Inconsistencies:

### Description Standardization:

Let's check if there are still stock codes that correspond to multiple descriptions:

In [25]:
stockcode_descriptions = df.groupby('StockCode')['Description'].unique()
diverse_stockcode_descriptions = stockcode_descriptions[stockcode_descriptions.str.len() > 1]
diverse_stockcode_descriptions_rows = df[df['StockCode'].isin(diverse_stockcode_descriptions.index)]
print('Number of unique stock codes that point to more than one description:', len(diverse_stockcode_descriptions))
print('Proportion of unique stock codes that point to more than one description:', proportion(diverse_stockcode_descriptions, stockcode_descriptions))
print('Transactions with stock codes that point to more than one description:', len(diverse_stockcode_descriptions_rows))
print('Proportion of transactions with stock codes that point to more than one description:', proportion(diverse_stockcode_descriptions_rows, df))

Number of unique stock codes that point to more than one description: 220
Proportion of unique stock codes that point to more than one description: 5.62%
Transactions with stock codes that point to more than one description: 48194
Proportion of transactions with stock codes that point to more than one description: 9.07%


~6% of the unique stock codes point to more than one description.\
\~9% of the transactions have a stock code that points to more than one description.\
Let's check a sample of what descriptions do those stock codes point to:

In [26]:
diverse_stockcode_descriptions.sample(10)

StockCode
23524       [HORSE & PONY WALL ART, WALL ART HORSE & PONY ]
23462         [ROCOCO WALL MIRROR WHITE, ROCOCO WALL MIROR]
84228     [HEN HOUSE W CHICK STANDING, HEN HOUSE WITH CH...
23543            [KEEP CALM WALL ART , WALL ART KEEP CALM ]
23537     [I LOVE LONDON WALL ART, WALL ART I LOVE LONDON ]
23324     [RUSTIC STRAWBERRY JAMPOT LARGE , RUSTIC STRAW...
23325     [RUSTIC STRAWBERRY JAMPOT SMALL, RUSTIC STRAWB...
22584     [PACK OF 6 PANNETONE GIFT BOXES, PACK OF 6 PAN...
23497     [CLASSIC CHROME BICYCLE BELL , CLASSIC CROME B...
16156L                      [WRAP, CAROUSEL, WRAP CAROUSEL]
Name: Description, dtype: object

The descriptions that have the same stock code seem to have the same meaning.\
We may assume that the most frequent description associated with each stockcode represents the canonical product name.\
So, let's standardize descriptions by replacing each with the most frequent description associated with each stockcode:

In [27]:
df = standardize_descriptions(df)

---

## Incorrect/Missing Values:

### Unspecified Countries:

Let's check what proportion of the data is represented by rows with an unspecified country.

In [28]:
unspecified_country_rows = df[df['Country'] == 'Unspecified']
print('Rows with transactions with an unspecified country:', len(unspecified_country_rows))
print('Proportion of transactions with an unspecified country:', proportion(unspecified_country_rows, df))

Rows with transactions with an unspecified country: 442
Proportion of transactions with an unspecified country: 0.08%


Only ~0.08% of the transactions contain an unspecified country. Let's check a sample of those transactions:

In [29]:
unspecified_country_rows.sample(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Cancelled
152714,549687,48116,DOORMAT MULTICOLOUR STRIPE,2,2011-04-11 13:29:00,7.95,12363.0,Unspecified,False
184531,552695,23078,ICE CREAM PEN LIP GLOSS,24,2011-05-10 15:31:00,1.25,16320.0,Unspecified,False
257260,559521,21888,BINGO SET,4,2011-07-08 16:26:00,3.75,NaN,Unspecified,False
233986,557499,23239,SET OF 4 KNICK KNACK TINS POPPIES,6,2011-06-20 15:25:00,4.15,16320.0,Unspecified,False
282785,561658,22733,3D TRADITIONAL CHRISTMAS STICKERS,2,2011-07-28 16:06:00,1.25,12743.0,Unspecified,False
184570,552695,21907,I'M ON HOLIDAY METAL SIGN,12,2011-05-10 15:31:00,2.10,16320.0,Unspecified,False
471669,576646,84997D,CHILDRENS CUTLERY POLKADOT PINK,4,2011-11-16 10:18:00,4.15,NaN,Unspecified,False
257288,559521,22260,FELT EGG COSY BLUE RABBIT,1,2011-07-08 16:26:00,0.85,NaN,Unspecified,False
257286,559521,20727,LUNCH BAG BLACK SKULL.,8,2011-07-08 16:26:00,1.65,NaN,Unspecified,False
282802,561658,22701,PINK DOG BOWL,1,2011-07-28 16:06:00,2.95,12743.0,Unspecified,False


They seem to be normal transactions.\
Since only ~0.08% of the transaction contain an unspecified country, they can be removed.

In [30]:
df = df[df['Country'] != 'Unspecified']

---

### Missing Descriptions:

Let's check what proportion of the data is represented by rows with a missing description.

In [31]:
missing_desc_rows = df[df['Description'].isna()]
print('Rows with a missing description:', len(missing_desc_rows))
print('Proportion of transactions with a missing description:', proportion(missing_desc_rows, df))

Rows with a missing description: 0
Proportion of transactions with a missing description: 0.0%


There seems to be no rows with missing description left as a result of previous cleaning processes.

---

### Missing Customer IDs:

Let's check what proportion of the data is represented by rows with a missing customer id.

In [32]:
missing_rows = df[df['CustomerID'].isna()]
print('Rows with a missing customer id:', len(missing_rows))
print('Proportion of transactions with a missing customer id:', proportion(missing_rows, df))

Rows with a missing customer id: 131315
Proportion of transactions with a missing customer id: 24.74%


~25% of the transactions don't have a customer id.\
The missing customer ids in the data are likely because of guest-ordering or incomplete order details which is normal for this kind of data.\
The rows with missing customer ids will be kept for non-customer-related EDA and will be removed when it's time for preprocessing for modeling.

---

# Exporting the Cleaned Dataset:

In [33]:
df.to_csv('data/interim/Online_Retail_cleaned.csv', index=False)

---